In [ ]:
import sys
import os
# Points the notebook to your poset_operad backend package
sys.path.append(os.path.abspath(os.path.join('..')))

from poset_operad.core.predicates import is_trivial_poset
print("Backend connected successfully!")


This notebook suite provides executable implementations of the algorithms developed in https://www.scienceopen.com/hosted-document?doi=10.14293/PR2199.003522.v2 (**Matrix Decomposition Algorithms for Accelerated Poset Isomorphism**). It serves as a computational companion to the paper, supporting reproducibility, experimentation, and further research on poset structure analysis.

The codebase is structured as a set of reusable functions, enabling users to explore, test, and build upon the decomposition techniques introduced in the paper.

In [ ]:
import numpy as np
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix
from collections import Counter
import itertools
import networkx as nx
from typing import Tuple
import hashlib
import time




In [ ]:
def is_partial_semi_equidualizable(M):
    """
    Determines if a poset matrix is partial semi-equidualizable by identifying
    saturated boundaries and verifying the disconnectivity of the structural core.

    Inputs:
        M (np.ndarray): An n x n binary adjacency matrix of a poset.

    Outputs:
        bool: True if saturated boundaries exist and the resulting core is
              disconnected; False otherwise.

    Logic Summary:
        1. Boundary Scan: Identifies semi-right depth (i) by counting leading columns
           saturated with 1s, and semi-left depth (j) by counting trailing
           saturated rows.
        2. Boundary Guard: If no depths are detected (i=0, j=0), the matrix
           cannot belong to this family.
        3. Core Extraction: Slices the matrix to isolate the inner principal
           submatrix (M_core) located between the identified depths.
        4. Disconnectivity Verification: Converts M_core into an undirected graph.
           If the graph contains more than one connected component, the
           definition is satisfied.

    Scientific Context:
        In the study of the Poset Operad, partial semi-equidualizability identifies
        a structural transition where a poset matrix possesses "universal" boundary
        elements (represented by saturated rows or columns). When these boundaries—
        termed semi-left and semi-right depths—are removed, the remaining principal
        submatrix (the core) must reveal a fundamentally disconnected architecture.
        This property is crucial for recursive decomposition, as it indicates that
        the poset is a specific type of operadic composition involving a linear
        order and a disjoint union of sub-posets.

    Complexity Analysis:
        - Time Complexity: O(n^2). Boundary scanning for depths i and j takes O(n^2)
          using vectorized checks. The connectivity verification of the core typically
          takes O(n + E) where E is the number of relations (E <= n^2) using graph
          traversal (BFS/DFS).
        - Space Complexity: O(n^2). Required for the internal graph representation
          during connectivity analysis and the storage of the extracted core submatrix.
    """
    n = M.shape[0]
    i, j = 0, 0

    # 1. Identify semi-right depth (saturated leading columns)
    while i < n and np.all(M[i:, i] == 1):
        i += 1

    # 2. Identify semi-left depth (saturated trailing rows)
    while j < n and np.all(M[n-j-1, :n-j] == 1):
        j += 1

    # Terminal condition: no boundaries detected
    if i == 0 and j == 0:
        return False

    # 3. Extract the principal core submatrix M_core
    if i > 0 and j > 0:
        # Ensure depths haven't consumed the entire matrix
        if i >= n - j: return False
        m_core = M[i:n-j, i:n-j]
    elif i > 0:
        m_core = M[i:, i:]
    else:
        m_core = M[:n-j, :n-j]

    # 4. Final check: verify the core is disconnected
    if m_core.size < 2:
        return False

    # Create an undirected graph to check for disjoint components
    G = nx.from_numpy_array(m_core)
    return not nx.is_connected(G)


In [ ]:
def extract_semiequidual_subcomponents(matrix):
    """
    Isolates the inner sub-matrix by stripping away saturated boundary layers.

    Inputs:
        matrix (np.ndarray): An n x n numerical or boolean array.

    Outputs:
        list: Contains the sliced [sub_matrix].
        str: Connectivity status (defaults to "connected").
        tuple: The calculated depths (d1, d2).

    Logic Summary:
        It calculates the 'depth' of leading columns (d1) and trailing rows (d2)
        that are fully saturated. It uses `np.argmin` on a boolean reduction to
        find the first index where the saturation breaks. It then returns a
        NumPy slice (view) of the matrix excluding these depths.

    Scientific Context:
        This serves as a kernel extraction method. By removing the 'equidual'
        outer layers, it isolates the non-trivial core of a system, often
        used in recursive matrix decomposition.

    Complexity Analysis:
        - Time: O(n²) to scan for depths.
        - Space: O(n²) for mask creation; O(1) for the output slice (view).
    """
    n = matrix.shape[0]
    if n == 0: return [matrix], "connected", (0, 0)
    d1_mask = (matrix != 0) | (np.arange(n)[:, None] < np.arange(n))
    d1_valid = np.all(d1_mask, axis=0)
    d1 = np.argmin(d1_valid) if not np.all(d1_valid) else n
    d2_mask = (matrix != 0) | (np.arange(n) > np.arange(n)[::-1][:, None])
    d2_valid = np.all(d2_mask, axis=1)[::-1]
    d2 = np.argmin(d2_valid) if not np.all(d2_valid) else n

    if d1 == 0 and d2 == 0:
        return [matrix], "connected", (0, 0)

    if d1 > 0 and d2 > 0 and d1 == d2:
        sub = matrix[d1:n-d2, d1:n-d2]
    elif d1 > 0:
        sub = matrix[d1:, d1:]
    else:
        sub = matrix[:n-d2, :n-d2]
    return [sub], "connected", (d1, d2)

def generate_semi_depth_equivalents(depths):
    """
    Generates alternative depth configurations that yield the same structural state.

    Inputs:
        depths (tuple): A pair of integers (d1, d2).

    Outputs:
        list: A unique list of tuples representing equivalent depth pairs.

    Logic Summary:
        The function applies geometric transformations to the depth pairs:
        it always includes the transpose (d2, d1), checks for midpoint
        splits if one depth is zero, and checks for expansion if depths are equal.

    Scientific Context:
        This explores the symmetry group of the equidualizable property.
        It identifies different boundary "thicknesses" that result in
        isomorphic inner kernels.

    Complexity Analysis:
        - Time: O(1) as it operates on a small, fixed number of integers.
        - Space: O(1) for the output list.
    """
    d1, d2 = depths
    equivs = [(d1, d2), (d2, d1)]
    if d1 == 0 or d2 == 0:
        val = max(d1, d2)
        if val % 2 == 0: equivs.append((val//2, val//2))
    elif d1 == d2:
        equivs.extend([(d1*2, 0), (0, d2*2)])
    return list(set(equivs))

def compute_triangular_saturation_metrics(matrix):
    """
    Quantifies the number of rows and columns fully satisfying a triangular mask.

    Inputs:
        matrix (np.ndarray): An n x n numerical or boolean array.

    Outputs:
        tuple: (int: saturated_rows, int: saturated_cols).

    Logic Summary:
        The function generates a lower triangular boolean mask. It then
        performs an element-wise logical OR between the matrix values and the
        inverse of the mask. Finally, it counts how many rows/cols consist
        entirely of True values.

    Scientific Context:
        This metric determines the "triangularity" of a matrix. In numerical
        methods, this helps track the progress of elimination algorithms
        where the goal is to saturate specific triangular regions.

    Complexity Analysis:
        - Time: O(n²) for full matrix mask comparison.
        - Space: O(n²) to store the mask and intermediate boolean results.
    """
    n = matrix.shape[0]
    if n == 0: return 0, 0
    lower_tri = np.tril(np.ones((n, n), dtype=bool))
    cond = (matrix == 1) | (~lower_tri)
    rows = np.count_nonzero(np.all(cond, axis=1))
    cols = np.count_nonzero(np.all(cond, axis=0))
    return int(rows), int(cols)

def get_signature(matrix):
    """
    Creates a permutation-invariant hash signature of the matrix structure.

    Inputs:
        matrix (np.ndarray): A numerical array of any shape.

    Outputs:
        int: A hash value representing the matrix's structural signature.

    Logic Summary:
        The function sums the rows and columns, sorts these sums to ensure
        invariance to element ordering, and combines them with the matrix
        shape into a tuple which is then hashed.

    Scientific Context:
        This is a heuristic for detecting matrix isomorphism. If two matrices
        have different signatures, they cannot be identical under permutation.
        It is commonly used in graph theory to quickly filter non-isomorphic graphs.

    Complexity Analysis:
        - Time: O(n² + n log n) due to summation and sorting.
        - Space: O(n) to store the row and column sum vectors.
    """
    if matrix.size == 0: return hash(None)
    rs = tuple(sorted(np.sum(matrix, axis=1)))
    cs = tuple(sorted(np.sum(matrix, axis=0)))
    return hash((matrix.shape, rs, cs))


In [ ]:
def are_poset_structures_strictly_equal(list_a, list_b):
    """
    Verifies the structural and value-level identity between two nested collections
    of Partially Ordered Set (poset) matrices.

    Inputs:
        list_a (list | np.ndarray): The primary nested structure or array.
        list_b (list | np.ndarray): The comparison nested structure or array.

    Outputs:
        bool: True if the structures are isomorphic (identical nesting, array
              shapes, and value multisets); False otherwise.

    Logic Summary:
        The function employs a recursive "Signaturing" approach. For individual
        arrays, it creates a signature containing the shape and a byte-serialized
        sorted version of the elements (ensuring element-order invariance). For
        lists, it recursively collects signatures of children, sorts them to
        handle list-level permutations, and hashes the result. Finally, it uses
        a multiset (Counter) comparison at the top level to verify that both
        collections contain the same structural "ingredients."

    Scientific Context:
        In order theory and combinatorial topology, posets are often represented
        via adjacency or relation matrices. This function checks for **Structural
        Isomorphism** while allowing for spatial variance—meaning two posets are
        equal if they contain the same relations, even if those relations are
        stored in a different memory order or list sequence.

    Complexity Analysis:
        - Time: O(K * N² log N²), where K is the total number of arrays and N²
          is the elements per array. Sorting the array values is the bottleneck.
        - Space: O(K * N² + M), where M is the recursion depth. Memory is
          allocated to store byte-string signatures for each array to allow
          hashing and multiset comparison.
    """
    # 1. Handle Null/None cases (O(1))
    if list_a is None or list_b is None:
        return list_a == list_b

    # 2. Type enforcement (O(1))
    if type(list_a) != type(list_b):
        return False

    # 3. Base Case: Comparison of individual NumPy arrays
    if isinstance(list_a, np.ndarray):
        if list_a.shape != list_b.shape:
            return False
        return np.array_equal(np.sort(list_a, axis=None), np.sort(list_b, axis=None))

    # 4. Recursive Case: Comparison of List Structures
    if len(list_a) != len(list_b):
        return False

    def get_item_signature(item):
        """Generates a hashable structural signature for nested items."""
        if isinstance(item, np.ndarray):
            return (item.shape, np.sort(item, axis=None).tobytes())
        elif isinstance(item, list):
            return tuple(sorted([get_item_signature(sub) for sub in item], key=lambda x: str(x)))
        return hash(item)

    # 5. Multiset comparison of signatures (O(K))
    counts_a = Counter(get_item_signature(x) for x in list_a)
    counts_b = Counter(get_item_signature(x) for x in list_b)

    return counts_a == counts_b


In [ ]:
def get_satisfying_posets(collection, predicate_func):
    """
    Filters a nested collection to retrieve all poset matrices that satisfy a
    specific structural or algebraic condition.

    Inputs:
        collection (list | tuple | np.ndarray): A nested or flat structure
            containing potential poset matrices (as lists or arrays).
        predicate_func (str | callable): The test condition. Can be a function
            reference (e.g., is_partial_semi_equidualizable) or a string name
            of a function existing in the global namespace.

    Outputs:
        list[np.ndarray]: A flattened list of all matrices from the collection
            that returned True when evaluated by the predicate.

    Logic Summary:
        The function performs a depth-first recursive traversal of the input
        collection. It distinguishes between organizational containers (lists/tuples)
        and leaf nodes (matrices). Leaf nodes are cast into NumPy arrays and
        passed to the `predicate`. If successful, the matrix is added to a
        running result list. String-based predicates are resolved dynamically
        using the `globals()` registry.

    Scientific Context:
        In computational combinatorics, researchers often generate large
        libraries of poset structures. This function acts as a **Structural
        Sieve**, allowing for the extraction of specific sub-classes (e.g.,
        "Find all matrices in this dataset that are semi-equidualizable")
        without flattening the original hierarchical data manually.

    Complexity Analysis:
        - Time: O(M * T), where M is the total number of elements/matrices in
          the collection and T is the time complexity of the `predicate_func`
          (commonly O(n²) for matrix operations).
        - Space: O(D + S * n²), where D is the maximum nesting depth (stack space),
          S is the number of satisfying matrices, and n² is the size of each matrix.
    """
    if isinstance(predicate_func, str):
        predicate = globals().get(predicate_func)
    else:
        predicate = predicate_func

    if not predicate or not callable(predicate):
        raise ValueError(f"Predicate function '{predicate_func}' not found or not callable.")

    satisfying_matrices = []

    for item in collection:
        # Check if item is a container (nested list/tuple) but not a numpy matrix
        if isinstance(item, (list, tuple)) and not isinstance(item, np.ndarray):
            # Recursively find matrices and extend the current list with results
            satisfying_matrices.extend(get_satisfying_posets(item, predicate))
        else:
            # Treat item as a potential matrix
            matrix = np.array(item)
            if predicate(matrix):
                satisfying_matrices.append(matrix)

    return satisfying_matrices


In [ ]:
def count_satisfying_posets(collection, predicate_func):
    """
    Quantifies the occurrences of matrices within a nested collection that satisfy
    a specified predicate function.

    Inputs:
        collection (list | tuple | np.ndarray): A nested or flat collection
            containing poset matrices (as lists or NumPy arrays).
        predicate_func (str | callable): The evaluation condition. Accepts a
            function reference or a string identifier for a global function.

    Outputs:
        int: The total count of matrices that returned True under the predicate.

    Logic Summary:
        The function utilizes a recursive depth-first traversal to navigate
        arbitrary nesting levels. It identifies leaf-node matrices, standardizes
        them as NumPy arrays, and evaluates them against the predicate. Unlike
        retrieval functions, this maintains a scalar accumulator (`total_count`)
        to minimize memory overhead by avoiding the storage of the matrices themselves.

    Scientific Context:
        In statistical combinatorics and poset theory, this function is used to
        calculate the **Density** or **Frequency** of specific structural
        properties within a generated population. It is essential for verifying
        theoretical distributions of properties like equidualizability across
        large-scale structural datasets.

    Complexity Analysis:
        - Time: O(M * T), where M is the total number of matrices and T is the
          computational cost of the predicate.
        - Space: O(L + n²), where L is the maximum recursion depth (stack) and
          n² is the size of the single matrix currently being evaluated. This
          is significantly more memory-efficient than retrieval, as it operates
          in O(1) space relative to the number of satisfying results.
    """
    if isinstance(predicate_func, str):
        predicate = globals().get(predicate_func)
    else:
        predicate = predicate_func

    if not predicate or not callable(predicate):
        raise ValueError(f"Predicate function '{predicate_func}' not found or not callable.")

    total_count = 0

    for item in collection:
        # Check if item is a container but not the terminal matrix
        if isinstance(item, (list, tuple)) and not isinstance(item, np.ndarray):
            total_count += count_satisfying_posets(item, predicate)
        else:
            # Cast to array to ensure .shape and boolean logic availability
            matrix = np.array(item)
            if predicate(matrix):
                total_count += 1

    return total_count


In [ ]:
def is_non_partial_semi_equidualizable(poset_matrix):
    """
    Determines if a matrix lacks any partial semi-equidualizable characteristics.

    Inputs:
        poset_matrix (np.ndarray): An n x n numerical or boolean array representing
            a partially ordered set.

    Outputs:
        bool: True if the matrix is NOT partially semi-equidualizable; False otherwise.

    Logic Summary:
        This function acts as a logical negation (NOT) of the
        `is_partial_semi_equidualizable` check. It evaluates the matrix and
        returns True only if no boundary saturation (row or column) is detected
        at any depth. Note: The implementation requires calling the
        `is_partial_semi_equidualizable` function with the provided matrix.

    Scientific Context:
        In structural classification, this identifies "irreducible" matrices
        that do not possess the standard boundary symmetries required for
        semi-equidualization. It isolates the subset of posets that cannot be
        simplified using triangular boundary reduction.

    Complexity Analysis:
        - Time: O(n²) inherited from the underlying boundary saturation scan.
        - Space: O(n²) for the temporary masks generated during the primary check.
    """
    return not is_partial_semi_equidualizable(poset_matrix)


In [ ]:
def update_nested_posets(collection, func):
    """
    Recursively traverses a nested structure to transform 2D NumPy arrays based on a
    provided transformation function.

    Inputs:
        collection (list | tuple | np.ndarray): A nested or flat collection potentially
            containing 2D NumPy arrays representing posets.
        func (callable): A transformation function that accepts a 2D np.ndarray and
            returns a list of replacement structures.

    Outputs:
        list | tuple | np.ndarray: The modified collection where specific 2D arrays
            have been replaced by the output of `func`, preserving the original
            nesting types (list/tuple).

    Logic Summary:
        The function uses recursive case-handling:
        1. Base Case: If it hits a 2D NumPy array, it applies `func`. If `func` returns
           a non-empty list, that list replaces the array; otherwise, the array is kept.
        2. Recursive Case: If it hits a list or tuple, it maps itself over the elements,
           maintaining the container type.
        3. Fallback: Returns the item as-is if it matches no specific criteria.

    Scientific Context:
        In hierarchical poset decomposition, nested structures represent
        evolutionary stages or sub-component relationships. This function acts as an
        **Evolutionary Operator**, allowing specific nodes (matrices) to "split" into
        sub-structures or be refined based on structural criteria like equidualization
        depths.

    Complexity Analysis:
        - Time: O(N * T), where N is the total count of elements and containers in
          the hierarchy, and T is the execution time of the transformation `func`.
        - Space: O(D + M), where D is the maximum recursion depth (stack) and M is
          the memory required for the newly generated list structures and array views.
    """
    if isinstance(collection, np.ndarray) and collection.ndim == 2:
        result = func(collection)
        if isinstance(result, list) and len(result) > 0:
            return result
        return collection

    elif isinstance(collection, list):
        return [update_nested_posets(item, func) for item in collection]

    elif isinstance(collection, tuple):
        return tuple(update_nested_posets(item, func) for item in collection)

    return collection


In [ ]:
def is_chain_or_antichain(matrix):
    """
    Identifies if a poset matrix represents a total order (Chain) or an
    identity relation (Antichain).

    Inputs:
        matrix (np.ndarray): An n x n binary or numerical poset matrix.

    Outputs:
        bool: True if the structure matches a Chain or an Identity matrix.

    Logic Summary:
        The function leverages row-sum invariants to classify the matrix.
        It calculates the sum of each row using `np.sum(axis=1)`.
        1. Antichain Check: If all row sums are exactly 1, the matrix is an Identity
           matrix (representing no comparable elements other than self).
        2. Chain Check: If the sorted row sums perfectly match the sequence [1, 2, ..., n],
           the matrix is structurally equivalent to a total order (Chain).

    Scientific Context:
        In order theory, an **Antichain** is a subset where no two elements are
        comparable, while a **Chain** is a subset where every pair of elements
        is comparable. These represent the two extremes of poset density. This
        check is a prerequisite for calculating the **Width** (max antichain) or
        **Height** (max chain) of a poset.

    Complexity Analysis:
        - Time: O(n²) to compute the row sums (since every element must be read),
          plus O(n log n) for sorting the sums.
        - Space: O(n) to store the vector of row sums and the comparison range.
    """
    n = matrix.shape[0]
    row_sums = np.sum(matrix, axis=1)

    # 1. Identity (Antichain) Check
    if np.all(row_sums == 1):
        return True

    # 2. Chain Check (Permuted Lower Triangular)
    if np.array_equal(np.sort(row_sums), np.arange(1, n + 1)):
        return True

    return False


In [ ]:
def is_trivial_poset(poset_matrix):
    """
    Validates if the matrix represents the identity (trivial) element of the poset operad.

    Inputs:
        poset_matrix (np.ndarray): An n x n binary matrix to be evaluated.

    Outputs:
        bool: True if the matrix is a 1x1 array equal to [1], False otherwise.

    Logic Summary:
        The function performs a two-step validation: first, it verifies the
        dimensionality is exactly 1x1. Second, it extracts the single scalar value
        using `.item()` and compares it to 1. This approach is optimized for
        scalar extraction, avoiding the overhead of array-wide boolean logic.

    Scientific Context:
        In the category of posets (and specifically within the poset operad),
        the **Trivial Poset** is the identity object. It represents a
        set with exactly one element, where the only relation is the
        reflexive (x ≤ x). It serves as the base case for recursive structural
        operations and operadic compositions.

    Complexity Analysis:
        - Time: O(1). The operation is a constant-time metadata check and
          scalar comparison, regardless of the size of the input matrix.
        - Space: O(1). No auxiliary data structures or memory allocations
          are required.
    """
    return poset_matrix.shape == (1, 1) and poset_matrix.item() == 1


In [ ]:
def build_poset_decomposition_tree(root_matrix):
    """
    Recursively decomposes a poset matrix into a hierarchical tree of non-trivial
    connected components.

    Inputs:
        root_matrix (np.ndarray): The initial n x n poset matrix to be decomposed.

    Outputs:
        list[list[np.ndarray]]: A nested hierarchy where each inner list contains
            the matrices (components) discovered at that specific recursion depth.

    Logic Summary:
        The function implements an iterative breadth-first decomposition. In each
        cycle, it takes a "pool" of matrices and applies
        `decompose_dual_core_into_components` to extract sub-structures. The
        resulting components are stored in the `hierarchy`. To determine the next
        level, the function filters the current components, keeping only those
        classified as "non-trivial" (via `is_non_trivial_poset`). The process
        terminates when no further components are generated or all components
        are trivial.

    Scientific Context:
        This represents a **Structural Resolution** process. It breaks down a
        complex poset into its irreducible building blocks (atoms). By
        organizing these into a tree, it maps the genealogy of the poset,
        revealing how high-level dependencies are composed of simpler,
        connected sub-relations.

    Complexity Analysis:
        - Time: O(K * n²), where K is the total number of submatrices discovered
          across all levels. The dominant cost is the component decomposition
          performed on every matrix in the hierarchy.
        - Space: O(K * n²) to store the entire collection of matrices within
          the hierarchy structure.
    """
    hierarchy = []
    current_level_pool = [root_matrix]

    while current_level_pool:
        level_components = []
        for matrix in current_level_pool:
            # Assumes decompose_dual_core_into_components is defined in scope
            _, components = decompose_dual_core_into_components(matrix)
            level_components.extend(components)

        if not level_components:
            break

        hierarchy.append(level_components)

        # Filters for the next iteration; acts as the recursive exit condition
        current_level_pool = [m for m in level_components if is_non_trivial_poset(m)]

    return hierarchy


In [ ]:
def extract_poset_direct_sum_components(matrix):
    """
    Identifies and isolates direct sum components from the core region of a poset matrix
    by scanning for boundary saturation transitions.

    Inputs:
        matrix (np.ndarray): A square (N x N) poset adjacency matrix.

    Outputs:
        list[list[np.ndarray]] | None: A list containing groups of connected submatrices
            if the inner regions are disconnected; otherwise, None.

    Logic Summary:
        The function performs a dual-directional scan (Forward/Column and Backward/Row)
        to locate boundaries between 'semi-depth' (saturated) and 'non-semi-depth' regions.
        Once a transition is detected, it isolates the 'inner_region' and checks for
        connectivity using `is_disconnected_poset`. If the region is disconnected, it
        recursively extracts the individual direct sum components.

    Scientific Context:
        In the study of **Poset Decompositions**, a direct sum corresponds to a
        disjoint union of posets where no elements between the sets are comparable.
        This function identifies these independent parallel structures by stripping
        away the "linear" boundary dependencies (semi-depths) that often obscure
        the internal parallel architecture of the order.

    Complexity Analysis:
        - Time: O(N³) to O(N⁴). The scanning loops are O(N), but the internal
          `is_disconnected_poset` and `extract_direct_sum_components` calls typically
          require O(N²) to O(N³) depending on the underlying graph traversal
          (BFS/DFS) or transitive closure algorithm used.
        - Space: O(N²). Necessary for storing slices, sub-matrix copies, and
          auxiliary reachability matrices during connectivity analysis.
    """
    dim = matrix.shape[0]
    col_idx, non_semi_col_depth, semi_col_depth = 0, 0, 0
    row_idx, non_semi_row_depth, semi_row_depth = 0, 0, 0

    extracted_data = []
    has_found_semi_col = False
    has_found_semi_row = False

    # 1. Forward Scan: Iterate through columns
    while col_idx < dim:
        column_slice = matrix[col_idx:, col_idx]

        if not all(column_slice):
            if has_found_semi_col:
                total_depth_r = non_semi_col_depth + semi_col_depth
                inner_region = matrix[total_depth_r:, total_depth_r:]

                if is_disconnected_poset(inner_region):
                    components = extract_direct_sum_components(inner_region)
                    extracted_data.append(components)
                    break
                elif col_idx < dim:
                    col_idx += 1
                    has_found_semi_col = False
                    non_semi_col_depth += 1
                else:
                    break
            else:
                non_semi_col_depth += 1
                col_idx += 1
        else:
            semi_col_depth += 1
            col_idx += 1
            has_found_semi_col = True

    # 2. Backward Scan: Iterate through rows from bottom up
    while row_idx < dim:
        row_slice = matrix[dim - row_idx - 1, :dim - row_idx]

        if not all(row_slice):
            if has_found_semi_row:
                total_depth_l = non_semi_row_depth + semi_row_depth
                inner_region = matrix[:-total_depth_l, :-total_depth_l]

                if is_disconnected_poset(inner_region):
                    components = extract_direct_sum_components(inner_region)
                    extracted_data.append(components)
                    break
                elif row_idx < dim:
                    row_idx += 1
                    has_found_semi_row = False
                    non_semi_row_depth += 1
                else:
                    break
            else:
                non_semi_row_depth += 1
                row_idx += 1
        else:
            semi_row_depth += 1
            row_idx += 1
            has_found_semi_row = True

    return extracted_data if extracted_data else None


In [ ]:
def is_non_trivial_poset(matrix):
    """
    Checks if a poset matrix is neither a linear chain nor a discrete antichain.

    Inputs:
        matrix (np.ndarray): An n x n binary poset matrix.

    Outputs:
        bool: True if the matrix is neither a chain nor an antichain, False otherwise.

    Logic Summary:
        The function identifies 'non-trivial' species elements by calculating the
        distribution of relations (row sums). It returns False if the matrix matches
        the signature of an Antichain (all row sums equal to 1) or a Chain
        (sorted row sums equal to [1, 2, ..., n]). Otherwise, it returns True,
        indicating a partially ordered structure with more complex dependencies.

    Scientific Context:
        In the classification of poset species, Chains (total orders) and Antichains
        (identity relations) represent the two degenerate extremes of order density.
        A **Non-Trivial Poset** contains both comparable and incomparable elements
        (beyond self-reflexivity), making it the primary object of interest for
        structural decomposition and semi-equidualizable analysis.

    Complexity Analysis:
        - Time: O(n²) to compute row sums across all elements, plus O(n log n)
          for the sorting comparison. Using vectorized row sums is significantly
          more efficient than Python-level loops.
        - Space: O(n) to store the 1D array of row sums and the comparison range.
    """
    n = matrix.shape[0]
    if n == 0:
        return False

    # Calculate relations per element
    row_sums = np.sum(matrix, axis=1)

    # 1. Antichain Check: Every element relates only to itself
    if np.all(row_sums == 1):
        return False

    # 2. Chain Check: Every element is comparable, forming a total order
    if np.array_equal(np.sort(row_sums), np.arange(1, n + 1)):
        return False

    return True


In [ ]:
def decompose_dual_core_into_components(poset_matrix):
    """
    Decomposes the disconnected core of a dualizable poset into its connected components.

    Inputs:
        poset_matrix (np.ndarray): An n x n dualizable (semi-left, semi-right,
            or double) poset matrix.

    Outputs:
        tuple[list[tuple], list[np.ndarray]]:
            - paired_results: List of (original_matrix, component_submatrix) pairs
              for lineage tracking.
            - components: List of the standalone connected component submatrices.

    Logic Summary:
        The function follows a multi-stage structural reduction:
        1. Core Extraction: Isolates the internal 'core' of the poset using
           boundary depth analysis (via `extract_disconnected_core_with_depths`).
        2. Graph Conversion: Treats the disconnected core as an undirected graph
           using the `NetworkX` library.
        3. Component Identification: Finds isolated sets of vertices (components)
           that have no relations between them.
        4. Submatrix Extraction: Uses `get_principal_submatrix` to retrieve the
           specific rows and columns corresponding to each component as a new
           poset matrix.

    Scientific Context:
        In the operad of poset matrices, dualizable posets often possess an
        internal structure that is "disconnected," meaning it is a direct sum of
        smaller posets. This function serves as the **Component Decoupler**,
        mathematically separating these parallel dependencies so they can be
        analyzed or transformed individually without interference from the
        saturated boundary layers.

    Complexity Analysis:
        - Time: O(n²) to O(n³). The core extraction and submatrix slicing are
          O(n²). Finding connected components in the graph is linear relative
          to the number of vertices and relations O(V+E), where V=n and E ≤ n².
        - Space: O(n²) for storing the adjacency graph and the resulting
          list of component submatrices.
    """
    # 1. Extract the disconnected core using boundary depth-analysis
    # This identifies the inner region after stripping saturated layers
    core_extraction = extract_disconnected_core_with_depths(poset_matrix)
    if core_extraction is None:
        return [], []

    disconn_core = core_extraction[0]

    # 2. Identify connected components of the core via undirected graph analysis
    # NetworkX handles the traversal (BFS/DFS) to find disjoint sets of indices
    graph = nx.from_numpy_array(disconn_core)
    components_indices = [sorted(list(c)) for c in nx.connected_components(graph)]

    # 3. Extract each component as a principal submatrix
    # Only the elements belonging to a specific component are kept in each sub-matrix
    submatrices = [
        get_principal_submatrix(disconn_core, idx_list)
        for idx_list in components_indices
    ]

    # 4. Pair each submatrix with the original input for structural tracking
    paired_results = [(poset_matrix, sub) for sub in submatrices]

    return paired_results, submatrices


In [ ]:
def get_principal_submatrix(matrix, index_set):
    """
    Extracts the principal submatrix from a poset matrix based on a subset of indices.

    Inputs:
        matrix (np.ndarray): The source n x n square matrix.
        index_set (list | set | np.ndarray): The collection of row and column indices
            to preserve in the submatrix.

    Outputs:
        np.ndarray: A new square submatrix containing only the specified elements.

    Logic Summary:
        The function standardizes the input indices into a list and utilizes
        `np.ix_` to perform advanced indexing. This creates a cross-product of
        the index arrays, effectively "zooming in" on the intersection of the
        specified rows and columns while maintaining their relative order.

    Scientific Context:
        In matrix algebra and graph theory, a **Principal Submatrix** is
        obtained by removing the same set of rows and columns. In the context
        of posets, this operation extracts the induced sub-poset. It is
        fundamental for analyzing sub-structures, such as finding connected
        components or calculating the minors of a structural matrix.

    Complexity Analysis:
        - Time: O(k²), where k is the number of indices in the `index_set`.
          NumPy must copy the selected elements into a new memory buffer.
        - Space: O(k²) to allocate and store the resulting submatrix.
    """
    # Convert to list to ensure compatibility with NumPy indexing
    indices = list(index_set)

    # matrix[np.ix_(indices, indices)] is the fastest way to get a principal submatrix.
    return matrix[np.ix_(indices, indices)]


In [ ]:
def extract_disconnected_core_with_depths(poset_matrix):
    """
    Extracts the internal disconnected submatrix and identifies boundary depths.

    Inputs:
        poset_matrix (np.ndarray): An n x n binary poset matrix.

    Outputs:
        tuple[np.ndarray, dict] | None:
            - submatrix: The extracted principal disconnected core.
            - metadata: A dictionary mapping the dualizability type to its identified depth(s).
            Returns None if no disconnected core is identified.

    Logic Summary:
        The function identifies the "semi-depths" of the matrix—the number of consecutive
        leading columns (depth1) or trailing rows (depth2) that are fully saturated. It
        then prioritizes three extraction cases:
        1. Double Dualizable: Strips both depths if they are equal and symmetric.
        2. Semi-Right: Strips depth1 from the top-left and checks for core disconnection.
        3. Semi-Left: Strips depth2 from the bottom-right and checks for core disconnection.
        Disconnection is verified via `check_poset_connectivity`.

    Scientific Context:
        In poset operad theory, the **Disconnected Core** represents the non-trivial
        inner structure of a dualizable poset. By isolating this core, we separate
        the "linear" boundary dependencies from the "parallel" internal components.
        This is a critical step in the recursive decomposition of operadic elements
        into their irreducible constituents.

    Complexity Analysis:
        - Time: O(n²) total. Scanning for depths is O(n²), and connectivity checks
          via graph traversal (BFS/DFS) are O(V+E) where E is up to n².
        - Space: O(n²) for storing the extracted submatrix slice and the
          intermediate graph representation used during connectivity testing.
    """
    n = poset_matrix.shape[0]
    depth1, depth2 = 0, 0

    # Vectorized scan for initial depth (columns)
    for i in range(n):
        if np.all(poset_matrix[i:, i]):
            depth1 += 1
        else:
            break

    # Vectorized scan for terminal depth (rows)
    for j in range(n):
        row_idx = n - j - 1
        if np.all(poset_matrix[row_idx, :row_idx + 1]):
            depth2 += 1
        else:
            break

    # Case 1: Double Dualizable (Symmetric depths and equal boundary element counts)
    if depth1 > 0 and depth1 == depth2:
        if len(get_maximal_elements(poset_matrix)) == len(get_minimal_elements(poset_matrix)):
            core = poset_matrix[depth1:n-depth2, depth1:n-depth2]
            return core, {"depth1,depth2": (depth1, depth2)}

    # Case 2: Semi-Right Dualizable
    if depth1 > 0:
        sub = poset_matrix[depth1:, depth1:]
        if not check_poset_connectivity(sub):
            return sub, {"depth1": depth1}

    # Case 3: Semi-Left Dualizable
    if depth2 > 0:
        sub = poset_matrix[:-depth2, :-depth2]
        if not check_poset_connectivity(sub):
            return sub, {"depth2": depth2}

    return None


In [ ]:
def check_poset_connectivity(poset_matrix):
    """
    Determines if the poset represented by the matrix is graph-theoretically connected.

    Inputs:
        poset_matrix (np.ndarray): An n x n binary matrix representing the poset relations.

    Outputs:
        bool: True if the poset's underlying undirected graph is connected; False otherwise.

    Logic Summary:
        The function applies a tiered validation approach:
        1. Empty Case: Returns False for 0x0 matrices.
        2. Trivial Case: Uses `is_trivial_poset` to verify if a 1x1 matrix represents the
           identity element.
        3. Multi-element Case: Converts the adjacency matrix into an undirected graph
           using NetworkX. It then performs a traversal (BFS/DFS) to check if all nodes
           belong to a single connected component.

    Scientific Context:
        In order theory, a poset is **Connected** if its Hasse diagram is not the
        disjoint union of two or more posets. Connectivity is a fundamental topological
        property; a disconnected poset can be represented as a **Direct Sum** of its
        connected components, which is a key operation in poset operad composition.

    Complexity Analysis:
        - Time: O(n + E), where n is the number of elements and E is the number of
          relations. The bottleneck is the graph traversal to identify components.
        - Space: O(n + E) to build and store the adjacency list representation of
          the undirected graph.
    """
    n = poset_matrix.shape[0]

    # Handle empty case
    if n == 0:
        return False

    # Handle trivial (rank 1) case using the utility function
    if n == 1:
        return is_trivial_poset(poset_matrix)

    # For n > 1, treat as an undirected graph to check for a single component
    graph = nx.from_numpy_array(poset_matrix)
    return nx.is_connected(graph)


In [ ]:
def get_maximal_elements(posetmatrix):
    r"""
    Identifies the indices of all maximal elements within a poset matrix using
    optimized vectorization.

    Inputs:
        posetmatrix (np.ndarray): An n x n 2D NumPy array where $M_{ij} = 1$
            denotes the relation $x_i \le x_j$.

    Outputs:
        list[int]: A list of column indices corresponding to the maximal elements.

    Logic Summary:
        The function computes the sum of each column simultaneously using
        `np.sum(axis=0)`. It then applies `np.where` to find all indices where
         the column sum is exactly 1 (the reflexive relation). This replaces
        the Python-level loop with a single pass in C, significantly
        reducing overhead for large matrices.

    Scientific Context:
        In the Hasse diagram of a poset, the **Maximal Elements** are those
        at the highest level with no outgoing edges. In terms of relation
        density, they represent "sink" nodes in the directed acyclic graph
        representing the order.

    Complexity Analysis:
        - Time: O(n²) total operations. While the complexity remains quadratic
          due to the need to touch every element, the use of BLAS-optimized
          NumPy sums provides a massive constant-factor speedup over
          iterative list comprehensions.
        - Space: O(n) to store the column-sum vector and the index results.
    """
    if posetmatrix.size == 0:
        return []

    # Calculate all column sums in a single vectorized pass
    col_sums = np.sum(posetmatrix, axis=0)

    # Identify indices where the sum is exactly 1
    max_indices = np.where(col_sums == 1)[0]

    return max_indices.tolist()


In [ ]:
def get_minimal_elements(posetmatrix):
    r"""
    Identifies the indices of all minimal elements within a poset matrix using
    optimized vectorization.

    Inputs:
        posetmatrix (np.ndarray): An n x n 2D NumPy array where $M_{ij} = 1$
            denotes the relation $x_i \le x_j$.

    Outputs:
        list[int]: A list of row indices corresponding to the minimal elements.

    Logic Summary:
        An element $x_i$ is minimal if no other element precedes it (i.e., $x_k \le x_i$
        implies $k = i$). In the adjacency matrix, this manifests as a row containing
        exactly one '1' (the reflexive relation $x_i \le x_i$). The function
        vectorizes this check by computing all row sums via `np.sum(axis=1)` and
        extracting indices where the sum equals 1 using `np.where`.

    Scientific Context:
        In order theory, **Minimal Elements** are the "sources" of a poset's Hasse
        diagram. They represent elements with no predecessors. Identifying these
        is essential for calculating the **Depth** of an element and for
        performing topological sorts on the underlying directed acyclic graph.

    Complexity Analysis:
        - Time: O(n²) total operations. The calculation of row sums requires
          scanning the entire matrix, but NumPy's vectorized implementation
          executes at C-speed, bypassing Python loop overhead.
        - Space: O(n) to store the row-sum vector and the resulting index list.
    """
    if posetmatrix.size == 0:
        return []

    # Calculate all row sums in a single vectorized pass
    row_sums = np.sum(posetmatrix, axis=1)

    # Identify indices where the sum is exactly 1 (reflexive only)
    min_indices = np.where(row_sums == 1)[0]

    return min_indices.tolist()


In [ ]:
def get_poset_signature(matrix):
    """
    Generates a permutation-invariant structural hash for high-dimensional poset matrices.

    Inputs:
        matrix (np.ndarray): An n x n binary or numerical array representing
            the poset relations.

    Outputs:
        int: A 64-bit integer hash representing the matrix's structural signature.

    Logic Summary:
        The function identifies structural invariants by calculating the sums
        along both axes (rows and columns). It sorts these sum vectors to ensure
        that the same signature is generated regardless of the specific
        index ordering (permutation invariance). Finally, it combines the
        matrix shape and these sorted sum tuples into a single hashable object.

    Scientific Context:
        This function computes a **Structural Invariant** for the poset. In
        matrix theory, if two matrices are isomorphic (identical up to
        relabeling of elements), they must share the same distribution of
        row and column weights. This is an efficient heuristic for the
        **Matrix Isomorphism Problem**, allowing for rapid filtering of
        non-identical structures in large datasets.

    Complexity Analysis:
        - Time: O(n² + n log n). O(n²) is required for the full matrix reduction
          via `np.sum`. The O(n log n) cost is for sorting the resulting
          vectors. This is highly efficient for large matrices compared to
          full graph isomorphism algorithms.
        - Space: O(n) to store the intermediate row and column sum tuples.
    """
    if matrix.size == 0:
        return hash(None)

    # Vectorized summation across both dimensions
    row_sums = tuple(sorted(np.sum(matrix, axis=1)))
    col_sums = tuple(sorted(np.sum(matrix, axis=0)))

    # Hash the shape and sorted distributions to create the signature
    return hash((matrix.shape, row_sums, col_sums))


In [ ]:
def check_all_isomorphisms(testbag, predicate):
    r"""
    Performs an exhaustive pairwise isomorphism test across a collection of poset matrices.

    Inputs:
        testbag (list[np.ndarray]): A collection of 2D NumPy arrays (poset matrices)
            to be compared.
        predicate (callable): An isomorphism-testing function, such as
            `are_isomorphicPM888`, which accepts two matrices and returns a boolean.

    Outputs:
        dict[tuple(int, int), bool]: A dictionary where keys are coordinate pairs
            (index_x, index_y) and values are the boolean results of the predicate.

    Logic Summary:
        The function generates the Cartesian product of the testbag's indices using
        `itertools.product`. It iterates through every possible ordered pair $(x, y)$,
        retrieves the corresponding matrices from the `testbag`, and applies the
        `predicate`. This ensures a complete cross-comparison, including
        self-identity $(x, x)$ and reciprocal pairs $(x, y)$ and $(y, x)$.

    Scientific Context:
        In computational group theory and combinatorics, this creates an
        **Isomorphism Matrix** or adjacency relation for a species. It is used to
        partition a library of generated posets into **Isomorphism Classes**,
        ensuring that subsequent statistical analysis only counts structurally
        unique entities.

    Complexity Analysis:
        - Time: $O(n^2 \cdot T)$, where $n$ is the number of matrices in the testbag
          and $T$ is the complexity of the `predicate`. For $n$ matrices, there are
          exactly $n^2$ comparisons.
        - Space: $O(n^2)$ to store the dictionary of results.
    """
    n = len(testbag)
    results = {}

    # Exhaustive comparison of all ordered pairs in the collection
    for x, y in itertools.product(range(n), repeat=2):
        results[(x, y)] = predicate(testbag[x], testbag[y])

    return results


In [ ]:
def generate_semi_equidual_poset(n):
    """
    Generates a canonical semi-equidual matrix (lower triangular).
    Column 0 is saturated (Universal Minimum).
    """
    mat = np.zeros((n, n), dtype=int)
    # 1. Universal Minimum: All elements relate to element 0
    mat[:, 0] = 1
    # 2. Add random lower-triangular relations
    for i in range(1, n):
        mat[i, i] = 1 # Reflexivity
        for j in range(1, i):
            if np.random.rand() > 0.7: # Sparse core
                mat[i, j] = 1

    # 3. Ensure Transitivity (Warshall's Algorithm)
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if mat[i, k] and mat[k, j]:
                    mat[i, j] = 1
    return mat

def get_isomorphic_permutation(mat):
    """Permutes the core of the matrix to create an isomorphic pair."""
    n = mat.shape[0]
    p = np.arange(n)
    # Permute only nodes 1 to n-1 to keep it semi-equidual for Tier 1
    core_indices = np.arange(1, n)
    np.random.shuffle(core_indices)
    p[1:] = core_indices
    return mat[np.ix_(p, p)]





In [ ]:
# Global cache to store results: {(hash_a, hash_b): bool}
isomorphism_cache = {}

def get_matrix_hash(matrix):
    """Generates a deterministic stable hash using SHA-256 for matrix content."""
    return hashlib.sha256(matrix.tobytes()).hexdigest()

def verify_poset_isomorphism_hierarchical(matrix_a, matrix_b, depth=0):
    """
    Verifies structural isomorphism between two poset adjacency matrices via
    hierarchical decomposition and canonical tree cross-validation.

    Args:
        matrix_a (np.ndarray): Primary n x n poset adjacency matrix.
        matrix_b (np.ndarray): Secondary n x n poset adjacency matrix.
        depth (int): Current recursion level tracking decomposition height.

    Returns:
        bool: True if a structural bijection exists, False otherwise.
    """
    # 1. Memoization Lookup (Deterministic)
    h_a = get_matrix_hash(matrix_a)
    h_b = get_matrix_hash(matrix_b)
    pair_key = tuple(sorted((h_a, h_b)))

    if pair_key in isomorphism_cache:
        return isomorphism_cache[pair_key]

    result = False

    try:
        # 2. Immediate Global Invariants
        # If shapes or the total number of relations (sum) differ, they cannot be isomorphic.
        if matrix_a.shape != matrix_b.shape or np.sum(matrix_a) != np.sum(matrix_b):
            result = False

        # 3. Trivial Case Check
        elif is_trivial_poset(matrix_a) and is_trivial_poset(matrix_b):
            result = True

        else:
            # 4. Recursive Decomposition Matching
            # Attempt to strip semi-equidual boundaries to find the inner structural core.
            extract_a = extract_semiequidual_subcomponents(matrix_a)
            extract_b = extract_semiequidual_subcomponents(matrix_b)

            if extract_a and extract_b:
                # Unpack: (submatrices, type, depth_coords)
                depths_a, comps_a = extract_a[2], extract_a[0]
                depths_b, comps_b = extract_b[2], extract_b[0]

                list_a = comps_a if isinstance(comps_a, list) else [comps_a]
                list_b = comps_b if isinstance(comps_b, list) else [comps_b]

                # Verify sub-problem validity: matrix must shrink and counts must match
                if all(m.shape < matrix_a.shape for m in list_a) and len(list_a) == len(list_b):
                    equiv_b = generate_semi_depth_equivalents(depths_b)

                    if equiv_b is not None and depths_a in equiv_b:
                        # Sort by signature to align potential matches before deep check
                        list_a.sort(key=lambda m: get_poset_signature(m))
                        list_b.sort(key=lambda m: get_poset_signature(m))

                        def is_robust_match(sa, sb):
                            # Recursive isomorphism check
                            if not verify_poset_isomorphism_hierarchical(sa, sb, depth + 1):
                                return False
                            # Cross-verify using canonical tree structures
                            t_a = build_poset_decomposition_tree(sa)
                            t_b = build_poset_decomposition_tree(sb)
                            return are_poset_structures_strictly_equal(t_a, t_b)

                        if all(is_robust_match(sa, sb) for sa, sb in zip(list_a, list_b)):
                            result = True

            # 5. Final Fallback to Tree Identity
            # If decomposition is not applicable or inconclusive, compare full canonical trees.
            if not result:
                tree_a = build_poset_decomposition_tree(matrix_a)
                tree_b = build_poset_decomposition_tree(matrix_b)
                result = are_poset_structures_strictly_equal(tree_a, tree_b)

    except Exception:
        # Catch unexpected structural inconsistencies during decomposition
        result = False

    # Store in cache and return
    isomorphism_cache[pair_key] = result
    return result


In [ ]:
isobag1 = [

        np.array([[1, 0, 0, 0],
                  [0, 1, 0, 0],
                  [0, 0, 1, 0],
                  [1, 1, 1, 1]]),
        np.array([[1, 0, 0, 0],
                  [1, 1, 0, 0],
                  [1, 0, 1, 0],
                  [1, 0, 0, 1]])

]


In [ ]:
isobag2=[np.array([[1, 0, 0, 0],
          [1, 1, 0, 0],
          [1, 0, 1, 0],
          [1, 0, 1, 1]]),
   np.array([[1, 0, 0, 0],
          [0, 1, 0, 0],
          [0, 1, 1, 0],
          [1, 1, 1, 1]]),
   np.array([[1, 0, 0, 0],
          [1, 1, 0, 0],
          [1, 1, 1, 0],
          [1, 0, 0, 1]]),
   np.array([[1, 0, 0, 0],
          [1, 1, 0, 0],
          [0, 0, 1, 0],
          [1, 1, 1, 1]])]

In [ ]:
check_all_isomorphisms(isobag2,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True}

In [ ]:
isobag3=[np.array([[1, 0, 0, 0],
          [0, 1, 0, 0],
          [1, 1, 1, 0],
          [1, 1, 1, 1]]),
   np.array([[1, 0, 0, 0],
          [1, 1, 0, 0],
          [1, 1, 1, 0],
          [1, 1, 0, 1]]),
   np.array([[1, 0, 0, 0],
          [1, 1, 0, 0],
          [1, 0, 1, 0],
          [1, 1, 1, 1]])]

In [ ]:
check_all_isomorphisms(isobag3,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True}

In [ ]:
isobag4=[np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [0, 0, 0, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 0, 1, 1, 0, 0],
         [0, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 0, 0, 1, 0, 0],
         [1, 0, 0, 1, 1, 0],
         [1, 0, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 0, 1, 1, 0, 0],
         [1, 0, 0, 0, 1, 0],
         [1, 0, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [0, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 0, 0, 1, 0, 0],
         [1, 0, 0, 0, 1, 0],
         [1, 0, 0, 0, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 0, 0, 1, 0, 0],
         [1, 0, 0, 0, 1, 0],
         [1, 0, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [0, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]])]

In [ ]:
check_all_isomorphisms(isobag4,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (0, 6): True,
 (0, 7): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (1, 6): True,
 (1, 7): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (2, 6): True,
 (2, 7): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (3, 6): True,
 (3, 7): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (4, 6): True,
 (4, 7): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True,
 (5, 6): True,
 (5, 7): True,
 (6, 0): True,
 (6, 1): True,
 (6, 2): True,
 (6, 3): True,
 (6, 4): True,
 (6, 5): True,
 (6, 6): True,
 (6, 7): True,
 (7, 0): True,
 (7, 1): True,
 (7, 2): True,
 (7, 3): True,
 (7, 4): True,
 (7, 5): True,
 (7, 6): True,
 (7, 7): True}

In [ ]:
isobag5=[np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 0, 0, 1, 0],
         [1, 1, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 0, 1, 0],
         [1, 1, 0, 0, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 0, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 1, 1, 0],
         [1, 1, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]])]

In [ ]:
check_all_isomorphisms(isobag5,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True}

In [ ]:
iso_double=isobag4+isobag5

In [ ]:
check_all_isomorphisms(iso_double,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (0, 6): True,
 (0, 7): True,
 (0, 8): False,
 (0, 9): False,
 (0, 10): False,
 (0, 11): False,
 (0, 12): False,
 (0, 13): False,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (1, 6): True,
 (1, 7): True,
 (1, 8): False,
 (1, 9): False,
 (1, 10): False,
 (1, 11): False,
 (1, 12): False,
 (1, 13): False,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (2, 6): True,
 (2, 7): True,
 (2, 8): False,
 (2, 9): False,
 (2, 10): False,
 (2, 11): False,
 (2, 12): False,
 (2, 13): False,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (3, 6): True,
 (3, 7): True,
 (3, 8): False,
 (3, 9): False,
 (3, 10): False,
 (3, 11): False,
 (3, 12): False,
 (3, 13): False,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (4, 6): True,
 (4, 7): True,


In [ ]:
isobag7=[np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 0, 0, 1, 0, 0],
         [1, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 0, 0, 1, 0],
         [1, 1, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 1, 1, 0],
         [1, 1, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 0, 1, 0],
         [1, 1, 0, 0, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 0, 0, 1, 0, 0],
         [1, 0, 0, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 0, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 0, 1, 1, 0, 0],
         [1, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]])]

In [ ]:
check_all_isomorphisms(isobag7,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (0, 6): True,
 (0, 7): True,
 (0, 8): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (1, 6): True,
 (1, 7): True,
 (1, 8): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (2, 6): True,
 (2, 7): True,
 (2, 8): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (3, 6): True,
 (3, 7): True,
 (3, 8): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (4, 6): True,
 (4, 7): True,
 (4, 8): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True,
 (5, 6): True,
 (5, 7): True,
 (5, 8): True,
 (6, 0): True,
 (6, 1): True,
 (6, 2): True,
 (6, 3): True,
 (6, 4): True,
 (6, 5): True,
 (6, 6): True,
 (6, 7): True,
 (6, 8): True,
 (7, 0): True,
 (7, 1): True,
 (7, 2): True,
 (7, 3): T

In [ ]:
isobag8=[np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 1, 1, 0],
         [1, 1, 0, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 0, 1, 1, 0, 0],
         [1, 0, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [0, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]])]

In [ ]:
check_all_isomorphisms(isobag8,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True}

In [ ]:
isobag9=[np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 0, 1, 0],
         [1, 1, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 0, 0, 1, 0, 0],
         [1, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]])]

In [ ]:
check_all_isomorphisms(isobag9,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True}

In [ ]:
isobag10=[np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 0, 1, 1, 0, 0],
         [1, 0, 1, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 0, 0, 1, 0, 0],
         [1, 0, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [0, 1, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 1, 1, 0],
         [1, 1, 0, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 0, 1, 0],
         [1, 1, 0, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 0, 0, 0, 1]])]

In [ ]:
check_all_isomorphisms(isobag10,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (0, 6): True,
 (0, 7): True,
 (0, 8): True,
 (0, 9): True,
 (0, 10): True,
 (0, 11): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (1, 6): True,
 (1, 7): True,
 (1, 8): True,
 (1, 9): True,
 (1, 10): True,
 (1, 11): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (2, 6): True,
 (2, 7): True,
 (2, 8): True,
 (2, 9): True,
 (2, 10): True,
 (2, 11): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (3, 6): True,
 (3, 7): True,
 (3, 8): True,
 (3, 9): True,
 (3, 10): True,
 (3, 11): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (4, 6): True,
 (4, 7): True,
 (4, 8): True,
 (4, 9): True,
 (4, 10): True,
 (4, 11): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True,


In [ ]:
check_all_isomorphisms(isobag10,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (0, 6): True,
 (0, 7): True,
 (0, 8): True,
 (0, 9): True,
 (0, 10): True,
 (0, 11): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (1, 6): True,
 (1, 7): True,
 (1, 8): True,
 (1, 9): True,
 (1, 10): True,
 (1, 11): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (2, 6): True,
 (2, 7): True,
 (2, 8): True,
 (2, 9): True,
 (2, 10): True,
 (2, 11): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (3, 6): True,
 (3, 7): True,
 (3, 8): True,
 (3, 9): True,
 (3, 10): True,
 (3, 11): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (4, 6): True,
 (4, 7): True,
 (4, 8): True,
 (4, 9): True,
 (4, 10): True,
 (4, 11): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True,


In [ ]:
isobag12=[np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [0, 0, 0, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 0, 1, 1, 0, 0],
         [1, 0, 1, 1, 1, 0],
         [1, 0, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [0, 1, 1, 1, 0, 0],
         [0, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 0, 0, 0, 0, 1]])]

In [ ]:
check_all_isomorphisms(isobag12,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True}

In [ ]:
isobag21=[np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 0, 0, 1, 0],
         [1, 1, 0, 0, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 0, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 0, 0, 1, 0, 0],
         [1, 0, 0, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]])]

In [ ]:
check_all_isomorphisms(isobag21,verify_poset_isomorphism_hierarchical)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True}

In [ ]:
def is_disconnected_poset(poset_matrix):
    """
    Determines if the poset represented by the matrix is disconnected based on its
    underlying relational structure.

    Inputs:
        poset_matrix (np.ndarray): An n x n binary matrix representing the poset relations.

    Outputs:
        bool: True if the poset consists of at least two disjoint components;
              False if it is a single connected entity.

    Logic Summary:
        This function acts as a logical negation of `check_poset_connectivity`.
        It evaluates the matrix to determine if there are elements or groups
        of elements with no relations (direct or indirect) between them,
        thereby identifying a fragmented structure.

    Scientific Context:
        In order theory, a **Disconnected Poset** is a poset that can be
        represented as the **Direct Sum** of two or more non-empty posets.
        Identifying disconnection is a prerequisite for structural
        decomposition, as it indicates that the system can be analyzed
        as independent, parallel sub-orders.

    Complexity Analysis:
        - Time: O(n + E), inherited from `check_poset_connectivity`. This
          includes the time to construct the undirected graph and perform
          a traversal (BFS/DFS).
        - Space: O(n + E) for the internal graph representation and
          traversal metadata.
    """
    # Simply return the negation of the connectivity check
    return not check_poset_connectivity(poset_matrix)


In [ ]:
def extract_direct_sum_components(poset_matrix):
    """
    Extracts all connected principal submatrices from a disconnected
    poset matrix using weak connectivity analysis.

    Inputs:
        poset_matrix (np.ndarray): An n x n (0,1)-matrix representing a poset.

    Outputs:
        list[np.ndarray]: A list of connected principal submatrices representing
            the direct sum components of the original poset.

    Logic Summary:
        1. **Component Labeling**: Uses `scipy.sparse.csgraph.connected_components`
           with `connection='weak'` to group nodes. This treats the directed poset
           edges as undirected paths, identifying disjoint sub-orders.
        2. **Principal Submatrix Extraction**: For each component label, it
           retrieves the specific indices and applies `np.ix_` to slice the
           original matrix into a square, principal submatrix.

    Scientific Context:
        In the study of **series-parallel posets** and operadic structures,
        a direct sum corresponds to the disjoint union of posets where no
        element in one component is comparable to any element in another.
        This function identifies these independent parallel structures, effectively
        "unstacking" the blocks of the matrix representation.

    Complexity Analysis:
        - Time: O(n + E), where n is the number of elements and E is the number
          of relations. The `connected_components` function is highly efficient,
          using BFS/DFS internally.
        - Space: O(n²), as it returns a list of new matrix copies representing
          the extracted components.
    """
    # 1. Identify connected components (treating the graph as undirected)
    # connection='weak' ensures nodes i and j are in the same component
    # if there's any path between them, ignoring direction.
    n_components, labels = connected_components(
        poset_matrix,
        directed=True,
        connection='weak'
    )

    # 2. Extract each principal submatrix based on component labels
    components = []
    for i in range(n_components):
        # Identify indices belonging to the current component
        indices = np.where(labels == i)[0]

        # Extract the principal submatrix using NumPy cross-product indexing
        # This preserves the internal relational structure of the component.
        submatrix = poset_matrix[np.ix_(indices, indices)]
        components.append(submatrix)

    return components


In [ ]:
def extract_poset_direct_sum_components(matrix):
    """
    Efficiently isolates direct sum components by identifying transition points
    between saturated boundaries and disconnected inner cores.

    Inputs:
        matrix (np.ndarray): An N x N binary poset adjacency matrix.

    Outputs:
        list[list[np.ndarray]] | None: Groups of connected submatrices (including
            trivial components) if disconnection is found; otherwise, None.

    Logic Summary:
        1. **Vectorized Depth Search**: Uses `np.all` across slices to find the
           first column/row that breaks the semi-depth (saturated) pattern.
        2. **Transition Detection**: Instead of a linear scan, it identifies the
           exact slice where the boundary saturation ends and evaluates the
           resulting `inner_region`.
        3. **Direct Sum Extraction**: If the inner region is disconnected, it
           extracts all components. This naturally includes trivial components
           (1x1 matrices or simple chains) that exist as independent parallel
           branches in the poset structure.

    Scientific Context:
        Direct sum decomposition in the poset operad relies on identifying
        independent parallel sub-orders. In matrix form, these appear as
        block-diagonal structures within the "non-saturated" core. Handling
        trivial components is essential for a complete operadic decomposition,
        as they represent identity elements or basic linear orders in the system.

    Complexity Analysis:
        - Time: O(N³). Vectorization reduces the scan to O(N²), but the
          connectivity check (`is_disconnected_poset`) remains the bottleneck.
        - Space: O(N²) for submatrix views and component storage.
    """
    dim = matrix.shape[0]
    if dim == 0: return None

    extracted_data = []

    # 1. Vectorized Forward Boundary Analysis
    # We find the depth where column saturation [i:, i] stops
    col_bounds = np.array([np.all(matrix[i:, i]) for i in range(dim)])
    # Identify transitions from True (saturated) to False (inner region)
    transitions_f = np.where(col_bounds[:-1] & ~col_bounds[1:])[0]

    for t_idx in transitions_f:
        # Define inner region starting after the saturated boundary
        depth = t_idx + 1
        inner = matrix[depth:, depth:]
        if inner.size > 0 and is_disconnected_poset(inner):
            # extract_direct_sum_components handles both trivial and non-trivial
            extracted_data.append(extract_direct_sum_components(inner))
            break # Found the primary transition

    # 2. Vectorized Backward Boundary Analysis
    # We find the depth where row saturation [n-j-1, :n-j] stops
    row_bounds = np.array([np.all(matrix[dim-j-1, :dim-j]) for j in range(dim)])
    transitions_b = np.where(row_bounds[:-1] & ~row_bounds[1:])[0]

    for t_idx in transitions_b:
        depth = t_idx + 1
        inner = matrix[:-depth, :-depth]
        if inner.size > 0 and is_disconnected_poset(inner):
            extracted_data.append(extract_direct_sum_components(inner))
            break

    return extracted_data if extracted_data else None


In [ ]:
def verify_isomorphism_via_direct_sum_decomposition(M1, M2):
    """
    Verifies poset isomorphism by decomposing matrices into their direct sum
    components and comparing their recursive structural hierarchies.

    Inputs:
        M1 (np.ndarray): The first n x n binary poset adjacency matrix.
        M2 (np.ndarray): The second n x n binary poset adjacency matrix.

    Outputs:
        bool: True if the matrices are isomorphic based on their direct sum
              components and internal density; False otherwise.

    Logic Summary:
        1. **Component Extraction**: Identifies the parallel sub-structures
           (direct sums) of both matrices. If one matrix decomposes and the
           other does not, they are immediately non-isomorphic.
        2. **Tree Construction**: Recursively builds decomposition trees for
           every extracted component using `build_poset_decomposition_tree`.
        3. **Structural Alignment**: Compares the two resulting tree hierarchies
           for strict structural equality (nesting and shapes).
        4. **Density Invariant**: As a final check, it extracts non-reducible
           leaf nodes and compares their lower-triangular sums (entry counts)
           as a multiset to ensure internal relation density matches.

    Scientific Context:
        In combinatorial species and poset theory, many complex orders are
        constructed as the disjoint union (direct sum) of smaller, connected
        posets. This function solves the isomorphism problem for the
        **Direct Sum Operad**, reducing a global comparison to a set of
        parallel local comparisons, which is significantly more efficient
        for highly disconnected or series-parallel structures.

    Complexity Analysis:
        - Time: O(K * N³), where K is the number of components and N is the
          dimension. The bottleneck is the recursive extraction of direct
          sum components and subsequent tree building for each.
        - Space: O(K * N²) to store the resulting decomposition trees and
          intermediate component slices.
    """
    # Step 1: Extract direct sum components (submatrices)
    ds1 = extract_poset_direct_sum_components(M1)
    ds2 = extract_poset_direct_sum_components(M2)

    # Conclusive early exit if extraction fails for one but not both
    if (ds1 is None) != (ds2 is None):
        return False
    if ds1 is None and ds2 is None:
        return False

    # Step 2: Recursive Deep Comparison
    tree1 = update_nested_posets(ds1, build_poset_decomposition_tree)
    tree2 = update_nested_posets(ds2, build_poset_decomposition_tree)

    # Perform strict structural comparison of the resulting tree hierarchies
    if are_poset_structures_strictly_equal(tree1, tree2):
        # Final Verification: Check internal density via triangular saturation
        list1 = get_satisfying_posets(tree1, is_non_partial_semi_equidualizable)
        list2 = get_satisfying_posets(tree2, is_non_partial_semi_equidualizable)

        # Use multisets of triangular entry counts as a final canonical invariant
        counts1 = sorted([np.sum(np.tril(m)) for m in list1])
        counts2 = sorted([np.sum(np.tril(m)) for m in list2])
        return counts1 == counts2

    return False


In [ ]:
def extract_maximal_disconnected_submatrices(matrix: np.ndarray) -> list[np.ndarray]:
    """
    Extracts all maximal principal disconnected submatrices from a poset matrix.

    Inputs:
        matrix (np.ndarray): A square (N x N) binary adjacency matrix of a poset.

    Outputs:
        list[np.ndarray]: A list of principal submatrices that are disconnected
            and not contained within any larger disconnected principal submatrix.

    Logic Summary:
        1. **Interval Scanning**: Iterates through all possible contiguous principal
           intervals [i, j]. For each interval, it extracts the submatrix and
           checks for graph-theoretic disconnection (more than one component).
        2. **Maximality Filtering**: Compares all identified disconnected index
           sets. It removes any set that is a subset of another, ensuring only
           the "largest" possible disconnected structures are retained.
        3. **Extraction**: Returns the filtered index sets as a list of NumPy
           arrays, sorted by their original starting index for stability.

    Scientific Context:
        In the study of **Poset Operads** and structural decomposition, a
        disconnected principal submatrix represents a localized "parallel"
        relation within the order. Identifying **Maximal** disconnected
        submatrices is crucial for finding the most significant structural
        splits in the poset, serving as the basis for the **Substitution
        Decomposition** of the associated graph or order.

    Complexity Analysis:
        - Time: O(N⁴). There are O(N²) candidate intervals. For each, the
          connectivity check (BFS/DFS via `connected_components`) takes O(N²).
        - Space: O(N²) to store the resulting list of submatrices and the
          temporary undirected graph representation.
    """
    n = matrix.shape[0]
    if n < 2:
        return []

    def is_disconnected(sub_mat):
        """Standard check for Definition 1.1 using undirected components."""
        if sub_mat.shape[0] < 2:
            return False
        # Create undirected adjacency: relation exists if i->j OR j->i
        undirected = (sub_mat.astype(bool) | sub_mat.astype(bool).T)
        # csgraph components ignores self-loops by default
        n_components, _ = connected_components(csr_matrix(undirected), directed=False)
        return n_components > 1

    # 1. Identify all disconnected principal intervals [i, j]
    disconnected_intervals = []
    for i in range(n):
        for j in range(i + 1, n):
            indices = list(range(i, j + 1))
            sub = matrix[np.ix_(indices, indices)]
            if is_disconnected(sub):
                disconnected_intervals.append(set(indices))

    # 2. Filter for Maximality (Remove subsets)
    disconnected_intervals.sort(key=len, reverse=True)
    maximal_intervals = []

    for current_set in disconnected_intervals:
        is_subset = False
        for existing_set in maximal_intervals:
            if current_set.issubset(existing_set):
                is_subset = True
                break
        if not is_subset:
            maximal_intervals.append(current_set)

    # 3. Extract final submatrices based on maximal index sets
    maximal_intervals.sort(key=lambda s: min(s))

    results = []
    for idx_set in maximal_intervals:
        sorted_indices = sorted(list(idx_set))
        results.append(matrix[np.ix_(sorted_indices, sorted_indices)].astype(int))

    return results


In [ ]:
def compute_triangular_saturation_metrics(matrix: np.ndarray) -> Tuple[int, int]:
    """
    Calculates the number of rows and columns that are fully saturated (all ones)
    within the lower triangular region of the adjacency matrix.

    This metric identifies elements that maintain full connectivity with all
    preceding elements (rows) or all succeeding elements (columns) within the
    defined triangular bounds.

    Args:
        matrix (np.ndarray): A square (N x N) binary poset adjacency matrix.

    Returns:
        Tuple[int, int]: A tuple containing:
            - row_count: Quantity of rows 'i' where matrix[i, :i+1] are all 1s.
            - col_count: Quantity of columns 'j' where matrix[j:, j] are all 1s.

    Complexity:
        Time: O(N^2). The function performs a global masking operation and
              boolean reduction across the N x N grid. While this is theoretically
              the same complexity class as loops, the vectorization provides
              significant constant-factor speedups for large N.
        Space: O(N^2). Requires allocation of a boolean mask of size N x N.
    """
    n = matrix.shape[0]
    if n == 0:
        return 0, 0

    # 1. Create a Lower Triangular Mask (True for indices <= diagonal)
    # O(N^2) space/time
    lower_tri_mask = np.tril(np.ones((n, n), dtype=bool))

    # 2. Vectorized Verification
    # We want to check if 'matrix' is 1 WHERE 'lower_tri_mask' is True.
    # Logic: A position is valid if:
    #   a) It is in the upper triangle (we don't care, so essentially 'True'), OR
    #   b) It is in the lower triangle AND the matrix value is 1.
    #
    # This creates a condition matrix where False only exists if a
    # required '1' in the lower triangle is missing.
    condition_matrix = (matrix == 1) | (~lower_tri_mask)

    # 3. Aggregation
    # Check rows: Are all requirements met in row 'i'?
    row_count = np.sum(np.all(condition_matrix, axis=1))

    # Check cols: Are all requirements met in col 'j'?
    col_count = np.sum(np.all(condition_matrix, axis=0))

    return int(row_count), int(col_count)

# --- Verification Block ---
if __name__ == "__main__":
    # Test case from previous context
    tmat_test = np.array([
        [1, 0, 0, 0, 0, 0], # Row 0: [1] -> Pass
        [1, 1, 0, 0, 0, 0], # Row 1: [1, 1] -> Pass
        [1, 1, 1, 0, 0, 0], # Row 2: [1, 1, 1] -> Pass
        [1, 1, 1, 1, 0, 0], # Row 3: [1, 1, 1, 1] -> Pass
        [1, 1, 1, 1, 1, 0], # Row 4: [1, 1, 1, 1, 1] -> Pass
        [1, 1, 1, 0, 0, 1]  # Row 5: [1, 1, 1, 0, 0, 1] -> Fail (indices 3,4 are 0)
    ])

    r_cnt, c_cnt = compute_triangular_saturation_metrics(tmat_test)

    print(f"Input Matrix Shape: {tmat_test.shape}")
    print(f"Saturated Rows (Base Chain Density): {r_cnt}") # Expected: 5
    print(f"Saturated Cols (Dual Chain Density): {c_cnt}") # Depends on col structure


Input Matrix Shape: (6, 6)
Saturated Rows (Base Chain Density): 5
Saturated Cols (Dual Chain Density): 4


In [ ]:
def verify_isomorphism_via_maximal_disconnection_and_saturation(M1, M2):
    """
    Verifies poset isomorphism by comparing global triangular saturation metrics
    and decomposing matrices into their maximal disconnected submatrices.

    Inputs:
        M1 (np.ndarray): The first n x n binary poset adjacency matrix.
        M2 (np.ndarray): The second n x n binary poset adjacency matrix.

    Outputs:
        bool: True if the matrices are isomorphic based on maximal independent
              blocks and saturation invariants; False otherwise.

    Logic Summary:
        1. **Saturation Check**: Computes and compares the row and column
           saturation counts. This acts as a global structural invariant; if
           the "chain density" differs, the matrices cannot be isomorphic.
        2. **Maximal Partitioning**: Decomposes both matrices into their
           maximal disconnected principal submatrices. This step identifies the
           largest independent relational clusters within the poset.
        3. **Structural Identity**: Employs `are_poset_structures_strictly_equal`
           to verify that the resulting multisets of sub-blocks are
           isomorphic in shape, nesting, and value distribution.

    Scientific Context:
        This method targets posets with irregular or non-obvious direct sum
        structures. While Tier 2 handles explicit disjoint unions, this Tier
        identifies localized "parallel" clusters that may be embedded within
        larger structures. The saturation metrics provide a fast symmetry
        check, while maximal disconnection serves as a rigorous structural sieve.

    Complexity Analysis:
        - Time: O(N⁴), dominated by the maximal disconnection search which
          scans O(N²) intervals. The saturation check is O(N²).
        - Space: O(N²) to store the extracted maximal submatrices and
          intermediate connectivity masks.
    """
    # Step 1: Compare Global Triangular Saturation Metrics (Symmetry Check)
    metric1 = compute_triangular_saturation_metrics(M1)
    metric2 = compute_triangular_saturation_metrics(M2)

    # Sorting ensures order-invariant comparison of row/column saturation counts
    if sorted(metric1) != sorted(metric2):
        return False

    # Step 2: Partition into Maximal Disconnected Principal Submatrices
    output1 = extract_maximal_disconnected_submatrices(M1)
    output2 = extract_maximal_disconnected_submatrices(M2)

    # Step 3: Final Structural Identity Check
    return are_poset_structures_strictly_equal(output1, output2)


In [ ]:
isobag6=[np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [0, 1, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 1, 1, 0],
         [1, 1, 0, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 0, 1, 0],
         [1, 1, 0, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 0, 0, 0, 1]])]

In [ ]:
def generate_base_poset(size=10):
    """Generates a connected, non-trivial 10x10 lower-triangular poset."""
    mat = np.eye(size, dtype=int)
    # Add random relations to ensure connectivity but maintain DAG
    for i in range(1, size):
        mat[i, i-1] = 1 # Linear spine to ensure connectivity
        for j in range(i-1):
            if np.random.rand() > 0.7:
                mat[i, j] = 1
    # Transitive closure (Warshall's algorithm)
    for k in range(size):
        for i in range(size):
            for j in range(size):
                if mat[i, k] and mat[k, j]:
                    mat[i, j] = 1
    return mat

def create_direct_sum(component, count):
    """Creates a direct sum of 'count' copies of the base component."""
    c_size = component.shape[0]
    total_n = c_size * count
    large_matrix = np.zeros((total_n, total_n), dtype=int)
    for i in range(count):
        start, end = i * c_size, (i + 1) * c_size
        large_matrix[start:end, start:end] = component
    return large_matrix



In [ ]:
check_all_isomorphisms(isobag6, verify_isomorphism_via_direct_sum_decomposition)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (0, 6): True,
 (0, 7): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (1, 6): True,
 (1, 7): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (2, 6): True,
 (2, 7): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (3, 6): True,
 (3, 7): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (4, 6): True,
 (4, 7): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True,
 (5, 6): True,
 (5, 7): True,
 (6, 0): True,
 (6, 1): True,
 (6, 2): True,
 (6, 3): True,
 (6, 4): True,
 (6, 5): True,
 (6, 6): True,
 (6, 7): True,
 (7, 0): True,
 (7, 1): True,
 (7, 2): True,
 (7, 3): True,
 (7, 4): True,
 (7, 5): True,
 (7, 6): True,
 (7, 7): True}

In [ ]:
check_all_isomorphisms(isobag6,  verify_isomorphism_via_maximal_disconnection_and_saturation)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (0, 6): True,
 (0, 7): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (1, 6): True,
 (1, 7): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (2, 6): True,
 (2, 7): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (3, 6): True,
 (3, 7): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (4, 6): True,
 (4, 7): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True,
 (5, 6): True,
 (5, 7): True,
 (6, 0): True,
 (6, 1): True,
 (6, 2): True,
 (6, 3): True,
 (6, 4): True,
 (6, 5): True,
 (6, 6): True,
 (6, 7): True,
 (7, 0): True,
 (7, 1): True,
 (7, 2): True,
 (7, 3): True,
 (7, 4): True,
 (7, 5): True,
 (7, 6): True,
 (7, 7): True}

In [ ]:
def verify_isomorphism_via_maximal_disconnection_and_saturation(M1, M2):
    """
    Verifies poset isomorphism by comparing global triangular saturation metrics
    and decomposing matrices into their maximal disconnected submatrices.

    Inputs:
        M1 (np.ndarray): The first n x n binary poset adjacency matrix.
        M2 (np.ndarray): The second n x n binary poset adjacency matrix.

    Outputs:
        bool: True if the matrices are isomorphic based on maximal independent
              blocks and saturation invariants; False otherwise.

    Logic Summary:
        1. **Saturation Check**: Computes and compares the row and column
           saturation counts. This acts as a global structural invariant; if
           the "chain density" differs, the matrices cannot be isomorphic.
        2. **Maximal Partitioning**: Decomposes both matrices into their
           maximal disconnected principal submatrices. This step identifies the
           largest independent relational clusters within the poset.
        3. **Structural Identity**: Employs `are_poset_structures_strictly_equal`
           to verify that the resulting multisets of sub-blocks are
           isomorphic in shape, nesting, and value distribution.

    Scientific Context:
        This method targets posets with irregular or non-obvious direct sum
        structures. While Tier 2 handles explicit disjoint unions, this Tier
        identifies localized "parallel" clusters that may be embedded within
        larger structures. The saturation metrics provide a fast symmetry
        check, while maximal disconnection serves as a rigorous structural sieve.

    Complexity Analysis:
        - Time: O(N⁴), dominated by the maximal disconnection search which
          scans O(N²) intervals. The saturation check is O(N²).
        - Space: O(N²) to store the extracted maximal submatrices and
          intermediate connectivity masks.
    """
    # Step 1: Compare Global Triangular Saturation Metrics (Symmetry Check)
    metric1 = compute_triangular_saturation_metrics(M1)
    metric2 = compute_triangular_saturation_metrics(M2)

    # Sorting ensures order-invariant comparison of row/column saturation counts
    if sorted(metric1) != sorted(metric2):
        return False

    # Step 2: Partition into Maximal Disconnected Principal Submatrices
    output1 = extract_maximal_disconnected_submatrices(M1)
    output2 = extract_maximal_disconnected_submatrices(M2)

    # Step 3: Final Structural Identity Check
    return are_poset_structures_strictly_equal(output1, output2)


In [ ]:
isobag11=[np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]])]

In [ ]:
check_all_isomorphisms(isobag11,  verify_isomorphism_via_maximal_disconnection_and_saturation)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True}

In [ ]:
isobag13=[np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 0, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 0, 0, 1, 0],
         [1, 1, 0, 0, 1, 1]])]

In [ ]:
check_all_isomorphisms(isobag13, verify_isomorphism_via_direct_sum_decomposition)

{(0, 0): True, (0, 1): True, (1, 0): True, (1, 1): True}

In [ ]:
check_all_isomorphisms(isobag13,  verify_isomorphism_via_maximal_disconnection_and_saturation)

{(0, 0): True, (0, 1): True, (1, 0): True, (1, 1): True}

In [ ]:
check_all_isomorphisms(isobag13,  verify_isomorphism_via_maximal_disconnection_and_saturation)

{(0, 0): True, (0, 1): True, (1, 0): True, (1, 1): True}

In [ ]:
isobag14=[np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]])]

In [ ]:
check_all_isomorphisms(isobag14,  verify_isomorphism_via_maximal_disconnection_and_saturation)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True}

In [ ]:
check_all_isomorphisms(isobag14,  verify_isomorphism_via_maximal_disconnection_and_saturation)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True}

In [ ]:
isobag15=[np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]])]

In [ ]:
check_all_isomorphisms(isobag15,  verify_isomorphism_via_maximal_disconnection_and_saturation)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True}

In [ ]:

isobag16=[np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 0, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 0, 1, 1]])]

In [ ]:
check_all_isomorphisms(isobag16, verify_isomorphism_via_direct_sum_decomposition)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True}

In [ ]:
isobag17=[np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 0, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 0, 1, 1]])]

In [ ]:
isobag18=[np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 0, 1, 0],
         [1, 1, 0, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]])]

In [ ]:
check_all_isomorphisms(isobag18, verify_isomorphism_via_direct_sum_decomposition)

{(0, 0): True, (0, 1): True, (1, 0): True, (1, 1): True}

In [ ]:
check_all_isomorphisms(isobag18,  verify_isomorphism_via_maximal_disconnection_and_saturation)

{(0, 0): True, (0, 1): True, (1, 0): True, (1, 1): True}

In [ ]:
isobag19=[np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [0, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0, 0],
         [1, 1, 0, 1, 1, 0],
         [1, 1, 0, 1, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 1, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [1, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 0, 0, 0, 1]])]

In [ ]:
check_all_isomorphisms(isobag19, verify_isomorphism_via_direct_sum_decomposition)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True}

In [ ]:
check_all_isomorphisms(isobag19,  verify_isomorphism_via_maximal_disconnection_and_saturation)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True}

In [ ]:
isobag20=[np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 1, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 0, 0, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 0, 1, 0],
         [1, 1, 1, 0, 1, 1]]),
  np.array([[1, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0],
         [0, 0, 1, 0, 0, 0],
         [1, 1, 1, 1, 0, 0],
         [1, 1, 1, 1, 1, 0],
         [1, 1, 1, 0, 0, 1]])]

In [ ]:
check_all_isomorphisms(isobag20, verify_isomorphism_via_direct_sum_decomposition)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True}

In [ ]:
check_all_isomorphisms(isobag20,  verify_isomorphism_via_maximal_disconnection_and_saturation)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True}

In [ ]:
isobag10 = [
    # 0
    np.array([[1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0], [1, 0, 1, 1, 0, 0], [1, 0, 1, 0, 1, 0], [1, 1, 1, 1, 1, 1]]),
    # 1
    np.array([[1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 0, 1, 0, 0], [1, 0, 0, 0, 1, 0], [1, 1, 1, 1, 1, 1]]),
    # 2
    np.array([[1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 0, 1, 0, 0], [1, 1, 1, 1, 1, 0], [1, 1, 0, 0, 0, 1]]),
    # 3
    np.array([[1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0], [1, 0, 0, 1, 0, 0], [1, 0, 1, 1, 1, 0], [1, 1, 1, 1, 1, 1]]),
    # 4
    np.array([[1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [0, 1, 1, 0, 0, 0], [0, 1, 0, 1, 0, 0], [1, 1, 1, 1, 1, 0], [1, 1, 1, 1, 1, 1]]),
    # 5
    np.array([[1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0], [1, 1, 1, 1, 1, 0], [1, 1, 1, 1, 1, 1]]),
    # 6
    np.array([[1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0], [0, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 0], [1, 1, 1, 1, 1, 1]]),
    # 7
    np.array([[1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 0, 1, 0, 0], [1, 1, 0, 1, 1, 0], [1, 1, 0, 1, 0, 1]]),
    # 8
    np.array([[1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 0, 1, 0, 0], [1, 1, 0, 0, 1, 0], [1, 1, 0, 1, 1, 1]]),
    # 9
    np.array([[1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0], [1, 1, 1, 1, 0, 0], [1, 0, 0, 0, 1, 0], [1, 1, 1, 1, 1, 1]]),
    # 10
    np.array([[1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [1, 1, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0], [1, 1, 1, 1, 1, 0], [1, 1, 1, 1, 1, 1]]),
    # 11
    np.array([[1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0], [1, 1, 1, 0, 0, 0], [1, 1, 1, 1, 0, 0], [1, 1, 1, 0, 1, 0], [1, 1, 0, 0, 0, 1]])
]


In [ ]:
check_all_isomorphisms(isobag10,  verify_isomorphism_via_maximal_disconnection_and_saturation)

{(0, 0): True,
 (0, 1): True,
 (0, 2): True,
 (0, 3): True,
 (0, 4): True,
 (0, 5): True,
 (0, 6): True,
 (0, 7): True,
 (0, 8): True,
 (0, 9): True,
 (0, 10): True,
 (0, 11): True,
 (1, 0): True,
 (1, 1): True,
 (1, 2): True,
 (1, 3): True,
 (1, 4): True,
 (1, 5): True,
 (1, 6): True,
 (1, 7): True,
 (1, 8): True,
 (1, 9): True,
 (1, 10): True,
 (1, 11): True,
 (2, 0): True,
 (2, 1): True,
 (2, 2): True,
 (2, 3): True,
 (2, 4): True,
 (2, 5): True,
 (2, 6): True,
 (2, 7): True,
 (2, 8): True,
 (2, 9): True,
 (2, 10): True,
 (2, 11): True,
 (3, 0): True,
 (3, 1): True,
 (3, 2): True,
 (3, 3): True,
 (3, 4): True,
 (3, 5): True,
 (3, 6): True,
 (3, 7): True,
 (3, 8): True,
 (3, 9): True,
 (3, 10): True,
 (3, 11): True,
 (4, 0): True,
 (4, 1): True,
 (4, 2): True,
 (4, 3): True,
 (4, 4): True,
 (4, 5): True,
 (4, 6): True,
 (4, 7): True,
 (4, 8): True,
 (4, 9): True,
 (4, 10): True,
 (4, 11): True,
 (5, 0): True,
 (5, 1): True,
 (5, 2): True,
 (5, 3): True,
 (5, 4): True,
 (5, 5): True,
